In [1]:
model_name = "vit-ragdoll"


import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=33):
    return ti(stmt, globals=globals(), number=n) * 1000 / n



BENCHMARK_REPEAT=77
df = pd.DataFrame()

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

import torch
from torch import nn
from torchvision import models
import pandas as pd

MAGIC_NUM = 7777e-5

device = torch.device("cpu:0")
model = models.vit_b_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * MAGIC_NUM for k, v in model.state_dict().items()})
model = model.to(device)

df = pd.DataFrame()
!cpupower frequency-set --governor performance

In [ ]:
import gc
model_file_base = "vit.mlir"
strategy = "heuristic"
ragdoll_results = []
ragdoll_throughput = []
for bs in range(1, 37):
    model_file = model_file_base + ".bs{}".format(bs)
    source_file = model_file + ".{}".format(strategy)
    target_file = source_file + ".cpu.vmfb"
    """
    # gen model with specified batch-size
    !ragdoll-opt {model_file_base} --ragdoll-autodiff-prepare-batch-size=batchsize={bs} > {model_file}
    
    !ragdoll-opt {model_file}  \
    --canonicalize \
    --enable-cse-in-legalizer \
    --symbol-dce \
    --ragdoll-autodiff-vjp-public-functions='strategy=heuristic' \
    --ragdoll-autodiff-vjp \
    --inline \
    --ragdoll-autodiff-inline-function-call \
    --ragdoll-initialisation \
    --eliminate-empty-tensors \
    --ragdoll-legalise-to-iree-compatibility \
    --ragdoll-raise-linalg-to-tosa \
    --ragdoll-forward-func-removal \
    --canonicalize \
    --cse > {source_file}
    """
    !iree-compile {source_file} \
    -o {target_file} \
    --iree-hal-target-backends=llvm-cpu \
    --iree-llvmcpu-fail-on-out-of-bounds-stack-allocation=0 \
    --iree-opt-const-eval=1 \
    --iree-opt-const-expr-hoisting=1 \
    --iree-opt-numeric-precision-reduction=1 \
    --iree-llvmcpu-target-cpu-features=host \
    --iree-llvmcpu-enable-ukernels=all \
    --iree-llvmcpu-slp-vectorization=1 \
    --iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
    --iree-llvmcpu-target-triple=x86_64-pc-linux-elf
    
    
    #ragdoll_binary = load_executable(target_file)


    #try:
        #f1 = timeit("ragdoll_binary.forward(image_np_t)") / BENCHMARK_REPEAT
        #print('ragdoll-opt1-gpu-forward in timeit: ', f1)
    !iree-benchmark-module \
    --module={target_file} \
    --device=cuda \
    --function=dforward \
    --input={bs}x1000xf32 \
    --batch_size={BENCHMARK_REPEAT} \
    --benchmark_repetitions=1 \
    --batch_concurrency=1 \
    --benchmark_min_time=0.1s \
    --print_statistics=true
    """
    b1 = ragdoll_model_benchmark(
        target_file,
        "dforward",
        [(bs, 1000)],
        device='cpu',
        warmups=15,
        repetitions=BENCHMARK_REPEAT, 
        measure_count=1)
    b1 = np.mean(b1)

    b1 = timeit("ragdoll_binary.dforward(grad_np)") / BENCHMARK_REPEAT

    print("measuring #", bs)
    print(b1)
    ragdoll_results.append(b1)
    ragdoll_throughput.append(bs/np.mean(ragdoll_bench))
    """

In [ ]:
for bs in range(1, 27):
    dyn_model = torch.compile(model, backend="inductor")
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = dyn_model(image)
    grad = torch.randn_like(output)
    
    torch.cuda.reset_peak_memory_stats(device=device)
    before = torch.cuda.memory_allocated(device=device)
    print("measuring #", bs)
    baseline_f = timeit("dyn_model(image.to(device))", 333)
    baseline_b = timeit("torch.autograd.grad(output.to(device), [image.to(device)], grad.to(device), retain_graph=True)", 333)
    #print(baseline_f)
    print(baseline_b)

    # 记录操作后的峰值内存使用情况
    peak_memory = torch.cuda.max_memory_allocated(device=device)
    
    # 显示结果
    print(f"Memory used before operation: {before / (1024**2):.2f} MB")
    print(f"Peak memory usage: {peak_memory / (1024**2):.2f} MB")

In [ ]:
from timeit import timeit as ti
def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n
for bs in range(1, 27):
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = model(image)
    grad = torch.randn_like(output)
    torch.cuda.reset_peak_memory_stats(device=device)
    before = torch.cuda.memory_allocated(device=device)
    print("measuring #", bs)
    baseline_f = timeit("model(image)", 30)
    baseline_b = timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3)
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    #print(baseline_f)
    print(baseline_b)
    
    # 记录操作后的峰值内存使用情况
    peak_memory = torch.cuda.max_memory_allocated(device=device)
    
    # 显示结果
    print(f"Memory used before operation: {before / (1024**2):.2f} MB")
    print(f"Peak memory usage: {peak_memory / (1024**2):.2f} MB")

In [ ]:
!sudo cpupower frequency-set --governor powersave